# The Scenario: The Coffee Bean Sourcing Problem
A national coffee chain is preparing to launch a new "Signature Morning Blend" across all its retail stores next quarter. To create this specific flavor profile, the sourcing team buys raw coffee beans from three different global regions: Colombia, Ethiopia, and Sumatra.

Each region charges a different price per ton, and each region's beans possess a unique "acidity score" and "caffeine rating."

The Goal
The procurement team wants to minimize the total cost of purchasing and shipping the raw coffee beans needed for the launch.

The Decisions to Make
You need to tell the procurement team exactly how many tons of beans to order from each of the three regions.

The Business Rules (Constraints)
The sourcing plan must strictly follow these rules to ensure the coffee tastes right and arrives on time:

Total Demand: The company needs exactly 10,000 tons of blended coffee to meet the projected sales demand for the launch.

Flavor Profile (Proportions): To ensure the blend has its signature fruity notes, the Ethiopian beans must make up at least 30% of the total blend.

Supplier Capacity: Due to a recent drought and shipping container shortages, the supplier in Sumatra can provide an absolute maximum of 2,500 tons of beans.

Quality Standard (Acidity): The final mixed blend must have an average acidity score of 5.0 or lower so it doesn't upset customers' stomachs. (Colombia, Ethiopia, and Sumatra beans all have different individual acidity scores that will average out when mixed).

Quality Standard (Caffeine): The final blend must have an average caffeine rating of at least 8.5 to ensure it gives customers enough of a morning kick.

Physical Reality: We cannot order negative amounts of coffee from any supplier.

### Importing Required Packages and Libraries

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import autograd.numpy as anp
from autograd import grad, jacobian

### Creating Relevant Data

In [2]:
data = pd.DataFrame(columns = ["Region","Variable", "Cost_Per_Ton", "Acidity_Score", "Caffeine_Rating"], data = [["Colombia", "x1", 4000, 6.0, 8], ["Ethiopia", "x2", 5500, 4.5, 9.5], ["Sumatra", "x3", 3500, 4, 7.5]]) 

In [3]:
data

,Region,Variable,Cost_Per_Ton,Acidity_Score,Caffeine_Rating
0,Colombia,x1,4000,6.0,8.0
1,Ethiopia,x2,5500,4.5,9.5
2,Sumatra,x3,3500,4.0,7.5


### Writing Objective Function, Bounds, and Constraints in Mathematical Formulas

objective function: minimize cost  

cost = cost1 * x1 + cost2 * x2 + cos3 * x3

x1 = tons from region 1 (Colombia)  
x2 = tons from region 2 (Ethiopia)  
x3 = tons from region 3 (Sumatra)  

bound_1: x1 > 0  
bound_2: x2 > 0  
bound_3: x3 > 0  

constraint_1: x1 + x2 + x3 - 10000 = 0
constraint_2: x2 - 0.3*(x1+x2+x3) >= 0  
constraint_3: 2500 - x3 >= 0

Multiplying LHS and RHS by (x1+x2+x3) to ensure it is not in denominator since x1=x2=x3 could be passed as 0 by the solver causing denominator to be 0  

constraint_4: 5*(x1+x2+x3) - ((acid1*x1 + acid2*x2 + acid3*x3)) >= 0  

constraint_5: ((caff1*x1 + caff2*x2 + caff3*x3)) - 8.5 * (x1 + x2 + x3) >= 0


In [5]:
caff1 = data[data["Variable"] == "x1"]["Caffeine_Rating"].values[0]
caff2 = data[data["Variable"] == "x2"]["Caffeine_Rating"].values[0]
caff3 = data[data["Variable"] == "x3"]["Caffeine_Rating"].values[0]

In [6]:
acid1 = data[data["Variable"] == "x1"]["Acidity_Score"].values[0]
acid2 = data[data["Variable"] == "x2"]["Acidity_Score"].values[0]
acid3 = data[data["Variable"] == "x3"]["Acidity_Score"].values[0]

In [7]:
cost1 = data[data["Variable"] == "x1"]["Cost_Per_Ton"].values[0]
cost2 = data[data["Variable"] == "x2"]["Cost_Per_Ton"].values[0]
cost3 = data[data["Variable"] == "x3"]["Cost_Per_Ton"].values[0]

### Creating Objective and Constraint Functions, and Bounds List

In [8]:
def objective(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return x1*cost1 + x2*cost2 + x3*cost3    

In [9]:
bound1 = [0, None]
bound2 = [0, None]
bound3 = [0, 2500]

bounds = [bound1, bound2, bound3]

In [10]:
def constraint1(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return x1 + x2 + x3 - 10000

In [11]:
def constraint2(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return x2 - 0.3 * (x1 + x2 + x3)

In [12]:
def constraint3(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return 2500 - x3

In [13]:
def constraint4(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return 5 * (x1 + x2 + x3) - x1*acid1 - x2*acid2 - x3*acid3

In [14]:
def constraint5(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return x1*caff1 + x2*caff2 + x3*caff3 - 8.5 * (x1 + x2 + x3)

In [15]:
# Initial Guess
theta0 = [4000,3500,2500]

### Method 1: Using SciPy and Not Passing in Gradient and Jacobian into Solver

In [16]:
constraints1 = [
    {"type":"eq", "fun":constraint1},
    {"type":"ineq", "fun":constraint2},
    {"type":"ineq", "fun":constraint3},
    {"type":"ineq", "fun":constraint4},
    {"type":"ineq", "fun":constraint5}
]

In [17]:
sol1 = minimize(fun = objective, x0 = theta0, method = "SLSQP", bounds = bounds, constraints = constraints1)

In [18]:
print("The mimimum cost is {} dollars.".format(int(round(sol1.fun, 0))))
print("The optimum quantity of coffee beans needed from Colombia is {} tons".format(round(sol1.x[0],3)))
print("The optimum quantity of coffee beans needed from Ethiopia is {} tons".format(round(sol1.x[1],3)))
print("The optimum quantity of coffee beans needed from Sumatra is {} tons".format(round(sol1.x[2],3)))

The mimimum cost is 45000000 dollars.
The optimum quantity of coffee beans needed from Colombia is 3846.154 tons
The optimum quantity of coffee beans needed from Ethiopia is 4038.462 tons
The optimum quantity of coffee beans needed from Sumatra is 2115.385 tons


### Method 2: Using SciPy and Calculating the Gradient and Jacobian Myself 

In [19]:
def objective_gradient(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return np.array([cost1, cost2, cost3], dtype = float)

In [20]:
def constraint1_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return np.array([1, 1, 1], dtype = float)

def constraint2_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return np.array([-0.3, 0.7, -0.3], dtype = float)

def constraint3_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return np.array([0, 0, -1], dtype = float)

def constraint4_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return np.array([5 - acid1, 5 - acid2, 5 - acid3], dtype = float)

def constraint5_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return np.array([caff1 - 8.5, caff2 - 8.5, caff3 - 8.5], dtype = float)

In [21]:
constraints2 = [
    {"type":"eq", "fun":constraint1, "jac":constraint1_jacobian},
    {"type":"ineq", "fun":constraint2, "jac":constraint2_jacobian},
    {"type":"ineq", "fun":constraint3, "jac":constraint3_jacobian},
    {"type":"ineq", "fun":constraint4, "jac":constraint4_jacobian},
    {"type":"ineq", "fun":constraint5, "jac":constraint5_jacobian}
]

In [22]:
sol2 = minimize(fun = objective, x0 = theta0, method = "SLSQP", jac = objective_gradient, bounds = bounds, constraints = constraints2)

In [23]:
print("The mimimum cost is {} dollars.".format(int(round(sol2.fun, 0))))
print("The optimum quantity of coffee beans needed from Colombia is {} tons".format(round(sol2.x[0],3)))
print("The optimum quantity of coffee beans needed from Ethiopia is {} tons".format(round(sol2.x[1],3)))
print("The optimum quantity of coffee beans needed from Sumatra is {} tons".format(round(sol2.x[2],3)))

The mimimum cost is 45000000 dollars.
The optimum quantity of coffee beans needed from Colombia is 3846.154 tons
The optimum quantity of coffee beans needed from Ethiopia is 4038.462 tons
The optimum quantity of coffee beans needed from Sumatra is 2115.385 tons


### Method 3: Using SciPy and Autograd for Gradient and Jacobian Calculations

In [24]:
obj_gradient = grad(objective)
con1_jacobian = jacobian(constraint1)
con2_jacobian = jacobian(constraint2)
con3_jacobian = jacobian(constraint3)
con4_jacobian = jacobian(constraint4)
con5_jacobian = jacobian(constraint5)

In [25]:
constraints3 = [
    {"type":"eq", "fun":constraint1, "jac":con1_jacobian},
    {"type":"ineq", "fun":constraint2, "jac":con2_jacobian},
    {"type":"ineq", "fun":constraint3, "jac":con3_jacobian},
    {"type":"ineq", "fun":constraint4, "jac":con4_jacobian},
    {"type":"ineq", "fun":constraint5, "jac":con5_jacobian}
]


In [26]:
sol3 = minimize(fun = objective, x0 = theta0, method = "SLSQP", jac = obj_gradient, bounds = bounds, constraints = constraints3)

In [27]:
print("The mimimum cost is {} dollars.".format(int(round(sol3.fun, 0))))
print("The optimum quantity of coffee beans needed from Colombia is {} tons".format(round(sol3.x[0],3)))
print("The optimum quantity of coffee beans needed from Ethiopia is {} tons".format(round(sol3.x[1],3)))
print("The optimum quantity of coffee beans needed from Sumatra is {} tons".format(round(sol3.x[2],3)))

The mimimum cost is 45000000 dollars.
The optimum quantity of coffee beans needed from Colombia is 3846.154 tons
The optimum quantity of coffee beans needed from Ethiopia is 4038.462 tons
The optimum quantity of coffee beans needed from Sumatra is 2115.385 tons


### Method 4: Using CPLEX

In [28]:
from docplex.mp.model import Model

In [29]:
mdl = Model(name = "Coffee_Optimizer")
x1 = mdl.continuous_var(lb = 0, name = "Columbia_Qty_Tons")
x2 = mdl.continuous_var(lb = 0, name = "Ethiopia_Qty_Tons")
x3 = mdl.continuous_var(lb = 0, ub = 2500, name = "Sumatra_Qty_Tons")

mdl.add_constraint(x1 + x2 + x3 == 10000, ctname = "Total_Qty_Needed_Tons")
mdl.add_constraint(x2 - 0.3*(x1 + x2 + x3) >= 0, ctname = "Ethiopian_Minimum_Qty_Tons")
mdl.add_constraint(2500 - x3 >= 0, ctname = "Sumatra_Maximum_Qty_Tons")
mdl.add_constraint(5 * (x1 + x2 + x3) - x1*acid1 - x2*acid2 - x3*acid3 >= 0, ctname = "Acidity_Maximum")
mdl.add_constraint(x1*caff1 + x2*caff2 + x3*caff3 - 8.5 * (x1 + x2 + x3) >= 0, ctname = "Caffeine_Rating_Minimum")

mdl.minimize(x1*cost1 + x2*cost2 + x3*cost3)
sol4 = mdl.solve()

In [30]:
print("The mimimum cost is {} dollars.".format(int(round(sol4.get_objective_value(), 0))))
print("The optimum quantity of coffee beans needed from Colombia is {} tons".format(round(sol4.get_value(x1),3)))
print("The optimum quantity of coffee beans needed from Ethiopia is {} tons".format(round(sol4.get_value(x2),3)))
print("The optimum quantity of coffee beans needed from Sumatra is {} tons".format(round(sol4.get_value(x3),3)))

The mimimum cost is 45000000 dollars.
The optimum quantity of coffee beans needed from Colombia is 3333.333 tons
The optimum quantity of coffee beans needed from Ethiopia is 4166.667 tons
The optimum quantity of coffee beans needed from Sumatra is 2500.0 tons
